# Get phenotypes associated with a list of genes from high content screens
This notebook takes a list of gene symbols and queries the IDR
for phenotypes associated with the genes in high content screens.

### Install dependencies if required

The cell below will install dependencies if you choose to run the notebook in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb#recent=true).

In [1]:
import csv
import os
import pandas as pd
from tempfile import NamedTemporaryFile

import scipy
import numpy
from skimage import filters
import matplotlib.pyplot as plt

import requests
import tifffile

### Set up where to query and session

In [2]:
INDEX_PAGE = "https://idr.openmicroscopy.org/webclient/?experimenter=-1"

# create http session
with requests.Session() as session:
    request = requests.Request('GET', INDEX_PAGE)
    prepped = session.prepare_request(request)
    response = session.send(prepped)
    if response.status_code != 200:
        response.raise_for_status()

### Get list of genes to query either from file (one gene symbol per line)
or enter directly in list in notebook.

In [3]:
# uncomment the next two lines if you would rather read gene list
# in from a file
# with open('./includes/FiveExampleGenes.txt') as f:
#    genes = f.read().splitlines()

# comment out the next line if you have read in the gene list from a file
#genes = ['ASH2L', 'ash2', '85441']
genes = ['CDK5RAP2','CETN2']
# check the gene list has been read in
genes[:5]

['CDK5RAP2', 'CETN2']

### Set up base URLS so can use shorter variable names later on

In [8]:
SEARCH_URL = "https://idr.openmicroscopy.org/searchengine/api/v1/resources/image/search/?key={key}&value={value}"

### Find images for each gene specified
For each gene, search of images either in plates or datasets then search for phenotypes associated with the images.
The results are saved in a CSV file.

In [9]:
GENE_SYMBOL = "Gene Symbol"

#### Helper method
Parse the output of the json and save it into the CVS file.

In [21]:
def parse_annotation(writer, json_data, gene):

    for p in json_data:
        screen_name = p["screen_name"] if p["screen_name"] else "-"
        plate_name = p["plate_name"] if p["plate_name"] else "-"
        project_name = p["project_name"] if p["project_name"] else "-"
        dataset_name = p["dataset_name"] if p["dataset_name"] else "-"
        image_id = p['id']
        ontologies = []  # for ontology terms for a phenotype
        row = {}
        for v in p['key_values']:
            key = v['name']
            value = v['value']
            # if there are ontology mappings for the
            # phenotype, add them to the ontologies list
            ontList = ['Phenotype Term Name',
                       'Phenotype Term Accession',
                       'Phenotype Term Accession URL']
            
            if key == 'Phenotype':  # has phenotype
                row[key] = value  # so create row

            elif key in ontList:
                ontologies.extend([key, value])
        if row:
            if (len(ontologies) > 0):  # 1+ ontology mapping
                row.update({'Gene': gene,
                            'Screen': screen_name,
                            'Plate': plate_name,
                            'Image': image_id,
                            'Project' : project_name,
                            'Dataset': dataset_name})
                # we have the start of a row now
                # but we want to print out as many rows
                # as there are ontology mappings
                # so if there is mapping to 1 ontology term
                # print 1 row, if there are 2 ontology terms
                # print 2 rows etc
                numberOfRows = len(ontologies)/6
                # this is 3 pairs of ontology values per
                # mapping, add the ontology mappings and print
                n = 1
                while (n <= numberOfRows):
                    row.update({ontologies[0]: ontologies[1],
                                ontologies[2]: ontologies[3],
                                ontologies[4]: ontologies[5]})
                    # remove that set of ontology mappings
                    ontologies = ontologies[6:]
                    writer.writerow(row)
                    n = n + 1

#### Retrieve data 
A CSV file is first created in the ``home`` directory. The CSV file can then be downloaded to your local machine. To download it, click ``File > Open``, select the CSV file and open it, then click ``File > Download``.

In [9]:
home = os.path.expanduser("~")
csvfile = NamedTemporaryFile("w", delete=False, newline='', dir=home, suffix=".csv")
try:
    fieldnames = [
        'Gene', 'Screen', 'Plate', 'Project', 'Dataset', 'Image',
        'Phenotype', 'Phenotype Term Name', 'Phenotype Term Accession',
        'Phenotype Term Accession URL']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
            
    for gene in genes:
        url = SEARCH_URL.format(**{'key': GENE_SYMBOL, 'value': gene})
        json_data = session.get(url).json()
        parse_annotation(writer, json_data['results']['results'], gene)
         
finally:
    csvfile.close()

### Load output into a data frame
View a subset of the data.

In [10]:
df = pd.read_csv(csvfile.name)
df.head(60)

,Gene,Screen,Plate,Project,Dataset,Image,Phenotype,Phenotype Term Name,Phenotype Term Accession,Phenotype Term Accession URL
0,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_04,-,-,1554140,polylobed (automatic),polylobed nuclear phenotype,CMPO_0000357,http://www.ebi.ac.uk/cmpo/CMPO_0000357
1,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_04,-,-,1554140,dynamic changes (automatic),increased variability of nuclear shape in popu...,CMPO_0000345,http://www.ebi.ac.uk/cmpo/CMPO_0000345
2,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_05,-,-,1554429,polylobed (automatic),polylobed nuclear phenotype,CMPO_0000357,http://www.ebi.ac.uk/cmpo/CMPO_0000357
3,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_05,-,-,1554429,dynamic changes (automatic),increased variability of nuclear shape in popu...,CMPO_0000345,http://www.ebi.ac.uk/cmpo/CMPO_0000345
4,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_06,-,-,1554686,polylobed (automatic),polylobed nuclear phenotype,CMPO_0000357,http://www.ebi.ac.uk/cmpo/CMPO_0000357
5,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_06,-,-,1554686,dynamic changes (automatic),increased variability of nuclear shape in popu...,CMPO_0000345,http://www.ebi.ac.uk/cmpo/CMPO_0000345
6,CDK5RAP2,-,-,idr0021-lawo-pericentriolarmaterial/experiment...,CDK5RAP2-C,1884816,protein localized to centrosome,protein localized in centrosome phenotype,CMPO_0000425,http://www.ebi.ac.uk/cmpo/CMPO_0000425
7,CDK5RAP2,-,-,idr0021-lawo-pericentriolarmaterial/experiment...,CDK5RAP2-C,1884821,protein localized to centrosome,protein localized in centrosome phenotype,CMPO_0000425,http://www.ebi.ac.uk/cmpo/CMPO_0000425
8,CDK5RAP2,-,-,idr0021-lawo-pericentriolarmaterial/experiment...,CDK5RAP2-C,1884837,protein localized to centrosome,protein localized in centrosome phenotype,CMPO_0000425,http://www.ebi.ac.uk/cmpo/CMPO_0000425
9,CDK5RAP2,-,-,idr0021-lawo-pericentriolarmaterial/experiment...,CDK5RAP2-C,1884813,protein localized to centrosome,protein localized in centrosome phenotype,CMPO_0000425,http://www.ebi.ac.uk/cmpo/CMPO_0000425


#### Filter by a specified phenotype 

In [11]:
value = 'CMPO_0000357'
df_filtered = df[df['Phenotype Term Accession'] == value]
df_filtered.head()

,Gene,Screen,Plate,Project,Dataset,Image,Phenotype,Phenotype Term Name,Phenotype Term Accession,Phenotype Term Accession URL
0,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_04,-,-,1554140,polylobed (automatic),polylobed nuclear phenotype,CMPO_0000357,http://www.ebi.ac.uk/cmpo/CMPO_0000357
2,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_05,-,-,1554429,polylobed (automatic),polylobed nuclear phenotype,CMPO_0000357,http://www.ebi.ac.uk/cmpo/CMPO_0000357
4,CDK5RAP2,idr0013-neumann-mitocheck/screenA (6),LT0065_06,-,-,1554686,polylobed (automatic),polylobed nuclear phenotype,CMPO_0000357,http://www.ebi.ac.uk/cmpo/CMPO_0000357


### Select the image
Select the first image if any

In [12]:
image_id = -1
if ~df_filtered.empty:
    image_id = df_filtered.head(1)['Image'].values[0]
else:
    image_id = df.head(1)['Image'].values[0]
print(image_id)

1554140


### License (BSD 2-Clause)¶

Copyright (C) 2017-2026 University of Dundee. All Rights Reserved.

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution. THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.